In [1]:
# Resolve the data directory even when executed standalone
import os
import glob
import pandas as pd

DATA_DIR = os.path.join(os.getcwd(), 'Parquet')
if not os.path.exists(DATA_DIR):
    DATA_DIR = r'C:\Users\admin\OneDrive\Desktop\Pro-1\Parquet'
if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f'Parquet directory not found: {DATA_DIR}')

parquet_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.parquet')))
if not parquet_files:
    raise FileNotFoundError(f'No parquet files found in {DATA_DIR}')

orig_files = []
perf_files = []

for f in parquet_files:
    fname = os.path.basename(f)
    if 'svcg' in fname.lower() or 'time' in fname.lower() or 'perf' in fname.lower():
        perf_files.append(f)
    else:
        orig_files.append(f)

# Official Freddie Mac column sequences (fallback if your parquet files lack headers)
ORIG_COLS = [
    'credit_score', 'first_payment_date', 'first_time_homebuyer_flag', 'maturity_date',
    'msa', 'mi_pct', 'number_of_units', 'occupancy_status', 'oclv', 'dti', 'orig_upb',
    'oltv', 'orig_interest_rate', 'channel', 'ppm_flag', 'amortization_type', 'property_state',
    'property_type', 'postal_code', 'loan_sequence_number', 'loan_purpose', 'orig_loan_term',
    'number_of_borrowers', 'seller_name', 'servicer_name', 'super_conforming_flag',
    'pre_harp_loan_sequence_number', 'program_indicator', 'harp_indicator',
    'property_valuation_method', 'interest_only_indicator', 'mi_cancellation_indicator'
]

PERF_COLS = [
    'loan_sequence_number', 'monthly_reporting_period', 'current_actual_upb',
    'current_loan_delinquency_status', 'loan_age', 'remaining_months_to_legal_maturity',
    'defect_settlement_date', 'modification_flag', 'zero_balance_code',
    'zero_balance_effective_date', 'current_interest_rate', 'current_deferred_upb',
    'ddlpi', 'mi_recoveries', 'net_sales_proceeds', 'non_mi_recoveries', 'expenses',
    'legal_costs', 'maintenance_and_preservation_costs', 'taxes_and_insurance',
    'miscellaneous_expenses', 'actual_loss_calculation', 'modification_cost',
    'step_modification_flag', 'deferred_payment_plan', 'eltv', 'zero_balance_removal_upb',
    'delinquent_accrued_interest', 'delinquency_due_to_disaster', 'borrower_assistance_status_code',
    'current_month_modification_cost', 'interest_bearing_upb'
]

def standardize_columns(df, expected_cols):
    """Assigns standard Freddie Mac names if columns are generic or headerless."""
    df = df.copy()
    normalized = [str(c).strip().lower() for c in df.columns]
    if 'loan_sequence_number' in normalized:
        df.columns = normalized
        return df
    if len(df.columns) == len(expected_cols):
        df.columns = [str(c).strip().lower() for c in expected_cols]
    else:
        df.columns = [f'col_{i}' for i in range(df.shape[1])]
    return df

# Process Performance Files one by one to save RAM
perf_summaries = []

for pf in perf_files:
    print(f'Processing target from: {os.path.basename(pf)}...')
    df_p = pd.read_parquet(pf)
    df_p = standardize_columns(df_p, PERF_COLS)

    # Some Freddie Mac parquet exports omit headers and store the raw schema in column positions.
    if 'loan_sequence_number' not in [str(c).lower() for c in df_p.columns]:
        df_p = df_p.iloc[:, [0, 3, 8]].copy()
        df_p.columns = ['loan_sequence_number', 'current_loan_delinquency_status', 'zero_balance_code']
    else:
        df_p = df_p[['loan_sequence_number', 'current_loan_delinquency_status', 'zero_balance_code']].copy()
    
    # Clean delinquency status: 'RA' (REO Acquisition) -> 99, numeric strings -> int, others -> 0
    delinq_str = df_p['current_loan_delinquency_status'].astype(str).str.strip()
    delinq_num = pd.to_numeric(delinq_str.replace({'RA': '99', 'XX': '0'}), errors='coerce').fillna(0)
    
    # Clean zero_balance_code: 03 (Short Sale), 06 (Repurchase), 09 (REO), 15 (Note Sale) = Default
    zb_num = pd.to_numeric(df_p['zero_balance_code'], errors='coerce').fillna(0)
    is_bad_zb = zb_num.isin([3, 6, 9, 15]).astype(int)

    # Flag Bad = 1 if Delinquency >= 3 (90+ DPD) OR Default Zero Balance Code
    df_p['default_flag'] = ((delinq_num >= 3) | (is_bad_zb == 1)).astype(int)
    
    # Collapse to 1 row per loan_sequence_number
    loan_level_target = df_p.groupby('loan_sequence_number', as_index=False)['default_flag'].max()
    perf_summaries.append(loan_level_target)
    del df_p  # Free memory immediately

# Combine all performance summaries and take max per loan
target_df = pd.concat(perf_summaries, ignore_index=True)
target_df = target_df.groupby('loan_sequence_number', as_index=False)['default_flag'].max()

print(f'\nTotal Unique Loans in Performance Data: {len(target_df):,}')
print(f'Overall Bad Rate (Default %): {target_df['default_flag'].mean() * 100:.2f}%')


NameError: name 'perf_files' is not defined